In [2]:
import sys
sys.path.append('../')

%env MUJOCO_GL=egl

env: MUJOCO_GL=egl


In [37]:
import os
import jax
import tyro
import mediapy
import functools
import numpy as np
import mujoco
import mediapy as media

from pathlib import Path
from dataclasses import dataclass
from utils.networks import load_params

In [4]:
AGENT = "ppo"

In [22]:
from ppo import Args as ALGArgs
@dataclass
class Args(ALGArgs):
    env_id: str = 'creative-4-task1'
    
    folder_path: str = "checkpoints/"
    fps: int = 10
    num_envs: int = 1

In [23]:
args = Args()

In [24]:
np.random.seed(args.seed)
key = jax.random.PRNGKey(args.seed)
key, key_env, key_eval, key_policy, key_value = jax.random.split(key, 5)

In [25]:
from builderbench.env_utils import make_env
from utils.wrapper import wrap_env
env_class, default_config = make_env(args)
# env = wrap_env( env_class( config=default_config), default_config.episode_length )  
env = env_class( config=default_config)
reset_fn = jax.jit(env.reset)
step_fn = jax.jit(env.step)

action_size = env.action_size

from utils.evaluation import get_video, Evaluator

from ppo import PPONetworks, Actor, Value
ppo_network = PPONetworks( 
    policy_network = Actor(layer_sizes=args.policy_hidden_sizes + [action_size * 2]),
value_network = Value(layer_sizes=args.value_hidden_sizes  + [1]),
)

from ppo import make_inference_fn
make_policy = make_inference_fn(ppo_network)

In [26]:
params = load_params(f"../checkpoints/stnd-episode-len-direct-force-control__creative-4-task1__42__ppo__1763914576/params_50.pkl")

In [27]:
actor_params, _, normalize_params = params

jit_inference_fn = jax.jit(
                    make_policy(
                        {
                            'policy': actor_params, 
                            'normalizer': normalize_params,
                        },
                        deterministic=True,
                    )
                )

In [29]:
action

(Array([-0.14573516,  0.6521482 , -0.12029382,  0.1048155 , -0.52369577],      dtype=float32),
 {})

In [30]:
rollout = []
returns = []
env_state = reset_fn(key_env)
rollout.append(env_state)

for i in range(default_config.episode_length):

    key_policy, key = jax.random.split(key)
    action, _ = jit_inference_fn(env_state.obs, env_state.info["target_goal"], key_policy) 

    env_state = step_fn(env_state, action)
    rollout.append(env_state)

    returns.append( env_state.reward )

Module mujoco.mjx.warp.ffi 441c945 load on device 'cuda:0' took 0.66 ms  (cached)


In [34]:
camera = mujoco.MjvCamera()
camera.distance = 0.8
camera.lookat = np.array([0.4, 0.0 , 0.4])
camera.elevation = -30.0
camera.azimuth = 180

In [35]:
video_images = []
mocap_key = 'target_mocap'
for i in range(default_config.episode_length):
    if i % 2 == 0:
        video_images.append(
            env.render_from_info(
                rollout[i].data.qpos,
                rollout[i].data.qvel, 
                rollout[i].info[f'{mocap_key}_pos'],
                rollout[i].info[f'{mocap_key}_quat'],
                camera=camera,
            )
        )

In [38]:
media.show_video(video_images, fps=1.0 / env.dt / 2)

This browser does not support the video tag.


In [42]:
returns

[Array(0.8334696, dtype=float32),
 Array(0.8280091, dtype=float32),
 Array(0.83073705, dtype=float32),
 Array(0.83500916, dtype=float32),
 Array(0.8408237, dtype=float32),
 Array(0.8482434, dtype=float32),
 Array(0.85735583, dtype=float32),
 Array(0.8682576, dtype=float32),
 Array(0.8810442, dtype=float32),
 Array(0.8958216, dtype=float32),
 Array(0.91271776, dtype=float32),
 Array(0.93188035, dtype=float32),
 Array(0.9534437, dtype=float32),
 Array(0.9774622, dtype=float32),
 Array(1.0038016, dtype=float32),
 Array(1.0320574, dtype=float32),
 Array(1.0615997, dtype=float32),
 Array(1.0915397, dtype=float32),
 Array(1.1207417, dtype=float32),
 Array(1.148164, dtype=float32),
 Array(1.1724703, dtype=float32),
 Array(1.1914258, dtype=float32),
 Array(1.206805, dtype=float32),
 Array(1.2243046, dtype=float32),
 Array(1.2483852, dtype=float32),
 Array(1.2820667, dtype=float32),
 Array(1.3274945, dtype=float32),
 Array(1.3857825, dtype=float32),
 Array(1.4558022, dtype=float32),
 Array(1.52

In [41]:
default_config.episode_length

300